# 🏭 Fase 1: Análisis Exploratorio (EDA) - AI4I 2020 Predictive Maintenance

En este notebook exploraremos el dataset **AI4I 2020** provisto por el repositorio UCI. Este dataset contiene registros simulados de telemetría de una máquina de fresado industrial, con el objetivo de predecir fallos inminentes basados en temperatura, desgaste, fuerza (torque) y velocidad.

### Objetivos del EDA:
1. Entender la distribución de las variables operativas.
2. Analizar el desbalance de clases (los fallos son eventos raros).
3. Crear nuevas variables (Feature Engineering) que puedan tener poder predictivo para nuestros modelos posteriores (Autoencoder y XGBoost).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background') # Estilo industrial oscuro
sns.set_palette("viridis")

## 1. Carga y Exploración Inicial

In [ ]:
# Cargar dataset
df = pd.read_csv('../data/ai4i2020.csv')
print(f"Forma del dataset: {df.shape}")
df.head()

In [ ]:
# Limpieza básica: Eliminar columnas de ID que no aportan valor predictivo
df_clean = df.drop(['UDI', 'Product ID'], axis=1)
df_clean.info()

## 2. Análisis del Desbalance de Clases
La variable objetivo global es `Machine failure`. Adicionalmente tenemos los modos específicos: `TWF`, `HDF`, `PWF`, `OSF`, `RNF`.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x='Machine failure', palette=['#2ecc71', '#e74c3c'])
plt.title('Distribución de Estado de Máquina (Normal vs Falla)')
plt.ylabel('Cantidad de Registros')

fallas = df_clean['Machine failure'].sum()
print(f"Total de fallas: {fallas} ({fallas/len(df_clean)*100:.2f}%)")

In [ ]:
# Desglose por tipo de falla
falla_tipos = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
conteo_fallas = df_clean[falla_tipos].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=conteo_fallas.index, y=conteo_fallas.values, palette='magma')
plt.title('Distribución de Tipos de Falla')
plt.ylabel('Ocurrencias')
plt.show()

## 3. Feature Engineering (Ingeniería de Características)
Creamos variables derivadas con significado físico para la máquina.

In [ ]:
df_fe = df_clean.copy()

# 1. Delta Temperatura (Diferencia entre proceso y ambiente)
df_fe['delta_temp'] = df_fe['Process temperature [K]'] - df_fe['Air temperature [K]']

# 2. Potencia Mecánica Estimada (Torque * RPM)
# Fórmula real: P(kW) = Torque(Nm) * RPM / 9550
df_fe['power_kw'] = (df_fe['Torque [Nm]'] * df_fe['Rotational speed [rpm]']) / 9550

# 3. Interacción Desgaste-Torque (Esfuerzo)
df_fe['wear_torque'] = df_fe['Tool wear [min]'] * df_fe['Torque [Nm]']

df_fe[['delta_temp', 'power_kw', 'wear_torque']].describe()

## 4. Correlaciones
Verifiquemos cómo las nuevas features se relacionan con las fallas.

In [ ]:
numeric_df = df_fe.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Matriz de Correlación (Features Originales + Creadas)')
plt.show()

### Conclusiones del EDA:
1. Hay un claro **desbalance de clases** (~3.4% de fallas) que deberemos manejar con SMOTE o `scale_pos_weight` en XGBoost.
2. Las fallas de sobreesfuerzo (`OSF`), disipación de calor (`HDF`) y potencia (`PWF`) son las más comunes.
3. Features como `delta_temp` están altamente correlacionadas con la probabilidad de `HDF`.
4. El dataset está listo para ser utilizado en el entrenamiento del modelo de Clasificación (Siguiente Notebook).

In [ ]:
# Exportamos el dataset enriquecido para el próximo paso
df_fe.to_csv('../data/ai4i2020_fe.csv', index=False)
print("Dataset con features exportado exitosamente a 'data/ai4i2020_fe.csv'")